# LLM Model Evaluation with LangChain, LangSmith & OpenEvals

This notebook walks through **end-to-end LLM evaluation** using:

| Tool | Role |
|------|------|
| **LangChain** | Build the target application (RAG chain) |
| **LangSmith `evaluate()`** | Run batch experiments over a dataset |
| **OpenEvals** | Ready-made evaluators (exact match, LLM-as-judge) |
| **Custom evaluators** | Domain-specific checks (topic match, retrieval quality) |

We evaluate the project's **Complaint RAG** assistant on a small golden dataset.

### Evaluation types covered
1. **Deterministic** — exact match, keyword/topic checks (fast, free)
2. **Retrieval** — did we fetch the right complaint category?
3. **LLM-as-judge** — correctness & helpfulness via OpenEvals (needs `OPENAI_API_KEY`)

## 0. Setup

In [1]:
import os
import sys
import uuid
from pathlib import Path

import pandas as pd
from IPython.display import display
from dotenv import load_dotenv

# Project root (notebooks/ is one level down)
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

from src.complaint_rag import ComplaintRAG, build_demo_dataset

DATA_DIR = PROJECT_ROOT / "data" / "complaints"
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not any(DATA_DIR.glob("*.json")):
    build_demo_dataset(DATA_DIR, count=50)

rag = ComplaintRAG(DATA_DIR, top_k=3)
print(f"Loaded {len(rag.records)} complaint records from {DATA_DIR}")
print(f"OPENAI_API_KEY set: {bool(os.getenv('OPENAI_API_KEY'))}")

Loaded 200 complaint records from C:\Users\Admin\Downloads\Buiild-main\Buiild-main\data\complaints
OPENAI_API_KEY set: False


## 1. Build a golden evaluation dataset

Good evals start with **inputs + reference outputs** (ground truth).
Each example has:
- `inputs.question` — what the user asks
- `outputs.expected_topic` — keyword we expect in the answer
- `outputs.reference_answer` — ideal short answer for LLM-as-judge

In [2]:
GOLDEN_EXAMPLES = [
    {
        "inputs": {"question": "A customer was overcharged on their invoice. What should support do?"},
        "outputs": {
            "expected_topic": "refund",
            "reference_answer": "Investigate the billing history, confirm the overcharge, and issue a refund with a clear explanation to the customer.",
        },
    },
    {
        "inputs": {"question": "My package arrived two weeks late. How was this handled?"},
        "outputs": {
            "expected_topic": "replacement",
            "reference_answer": "Resend the shipment at no extra cost and provide proactive tracking updates to the customer.",
        },
    },
    {
        "inputs": {"question": "The product I received was defective. What resolution is typical?"},
        "outputs": {
            "expected_topic": "warranty",
            "reference_answer": "Replace the product under warranty and offer a prepaid return label.",
        },
    },
    {
        "inputs": {"question": "How do we handle subscription cancellation complaints?"},
        "outputs": {
            "expected_topic": "credit",
            "reference_answer": "Cancel the subscription and apply a service credit for billing confusion or inconvenience.",
        },
    },
    {
        "inputs": {"question": "A customer says support took too long to respond. What is the escalation path?"},
        "outputs": {
            "expected_topic": "priority",
            "reference_answer": "Escalate to the priority support queue and schedule a follow-up call within one business day.",
        },
    },
]

pd.DataFrame([
    {"question": ex["inputs"]["question"], "expected_topic": ex["outputs"]["expected_topic"]}
    for ex in GOLDEN_EXAMPLES
])

,question,expected_topic
0,A customer was overcharged on their invoice. W...,refund
1,My package arrived two weeks late. How was thi...,replacement
2,The product I received was defective. What res...,warranty
3,How do we handle subscription cancellation com...,credit
4,A customer says support took too long to respo...,priority


## 2. Define the target application

In LangSmith, the **target** is the function/system you are evaluating.
Here we wrap `ComplaintRAG.answer()` so it returns a consistent output schema.

In [3]:
def complaint_rag_target(inputs: dict) -> dict:
    """Target function for LangSmith evaluate()."""
    question = inputs["question"]
    result = rag.answer(question)
    return {
        "answer": result["answer"],
        "sources": result.get("sources", []),
    }

# Quick smoke test
sample = complaint_rag_target(GOLDEN_EXAMPLES[0]["inputs"])
print(sample["answer"][:300], "...")

Based on the retrieved complaints, here is a concise response:

Complaint: Customer reported a billing issue related to billing.
Suggested solution: Support resolved it with refund and documented the follow-up. The customer was overcharged and received a refund.

Complaint: Customer reported a billi ...


## 3. Deterministic evaluators (OpenEvals + custom)

**Deterministic evaluators** are rule-based — no LLM call, fast and reproducible.

- `openevals.exact.exact_match` — strict string equality
- Custom **topic keyword** evaluator — checks if expected topic appears in the answer
- Custom **retrieval relevance** — checks if top retrieved complaint mentions the topic

In [4]:
from openevals.exact import exact_match
from openevals.string import levenshtein_distance


def topic_keyword_evaluator(run, example):
    """Pass if the expected topic keyword appears in the model answer."""
    outputs = run.outputs or {}
    reference = example.outputs or {}
    answer = outputs.get("answer", "").lower()
    expected = reference.get("expected_topic", "").lower()
    passed = expected in answer
    return {
        "key": "topic_keyword",
        "score": 1.0 if passed else 0.0,
        "comment": f"Expected '{expected}' in answer" if not passed else "Topic found",
    }


def retrieval_relevance_evaluator(run, example):
    """Pass if retrieved sources relate to the expected topic."""
    outputs = run.outputs or {}
    reference = example.outputs or {}
    answer = outputs.get("answer", "").lower()
    expected = reference.get("expected_topic", "").lower()
    # RAG answer includes retrieved context; check complaint text mentions topic family
    topic_aliases = {
        "refund": ["refund", "billing", "overcharg"],
        "replacement": ["replacement", "delivery", "shipment", "resent"],
        "warranty": ["warranty", "defective", "product", "replaced"],
        "credit": ["credit", "subscription", "cancel"],
        "priority": ["priority", "escalat", "support", "follow-up"],
    }
    aliases = topic_aliases.get(expected, [expected])
    passed = any(alias in answer for alias in aliases)
    return {
        "key": "retrieval_relevance",
        "score": 1.0 if passed else 0.0,
        "comment": f"Checked aliases: {aliases}",
    }


def answer_length_evaluator(run, example):
    """Sanity check: answer should be substantive but not excessively long."""
    answer = (run.outputs or {}).get("answer", "")
    length = len(answer.split())
    score = 1.0 if 20 <= length <= 500 else 0.0
    return {"key": "answer_length", "score": score, "comment": f"Word count: {length}"}


# Demo deterministic evaluators on one example
demo_run_outputs = complaint_rag_target(GOLDEN_EXAMPLES[0]["inputs"])
from langsmith.schemas import Run

class SimpleRun:
    def __init__(self, outputs):
        self.outputs = outputs

class SimpleExample:
    def __init__(self, outputs):
        self.outputs = outputs

demo_run = SimpleRun(demo_run_outputs)
demo_example = SimpleExample(GOLDEN_EXAMPLES[0]["outputs"])

print("topic_keyword:", topic_keyword_evaluator(demo_run, demo_example))
print("retrieval_relevance:", retrieval_relevance_evaluator(demo_run, demo_example))
print("answer_length:", answer_length_evaluator(demo_run, demo_example))
print("levenshtein (demo):", levenshtein_distance(
    outputs={"answer": "refund issued"},
    reference_outputs={"answer": "refund issued"},
))

topic_keyword: {'key': 'topic_keyword', 'score': 1.0, 'comment': 'Topic found'}
retrieval_relevance: {'key': 'retrieval_relevance', 'score': 1.0, 'comment': "Checked aliases: ['refund', 'billing', 'overcharg']"}
answer_length: {'key': 'answer_length', 'score': 1.0, 'comment': 'Word count: 94'}
levenshtein (demo): {'key': 'levenshtein_distance', 'score': 1.0, 'comment': None, 'metadata': None}


## 4. Run the full evaluation with LangSmith `evaluate()`

`langsmith.evaluation.evaluate()` orchestrates the full loop:

```
for each example in dataset:
    1. Call target(inputs) → outputs
    2. Run each evaluator(outputs, reference_outputs) → scores
    3. Aggregate results into an experiment
```

We use `upload_results=False` so this runs **locally** without a LangSmith API key.

In [5]:
from langsmith.evaluation import evaluate
from langsmith.schemas import Example

langsmith_examples = [
    Example(
        id=uuid.uuid4(),
        inputs=ex["inputs"],
        outputs=ex["outputs"],
    )
    for ex in GOLDEN_EXAMPLES
]

DETERMINISTIC_EVALUATORS = [
    topic_keyword_evaluator,
    retrieval_relevance_evaluator,
    answer_length_evaluator,
]

experiment_results = evaluate(
    complaint_rag_target,
    data=langsmith_examples,
    evaluators=DETERMINISTIC_EVALUATORS,
    experiment_prefix="complaint-rag-deterministic",
    description="Deterministic evaluators on Complaint RAG golden set",
    upload_results=False,
    max_concurrency=2,
)

print(f"Experiment: {experiment_results}")

C:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\langsmith\evaluation\_runner.py:392: LangSmithBetaWarning: 'upload_results' parameter is in beta.
  _warn_once("'upload_results' parameter is in beta.")


Starting evaluation of experiment: %s complaint-rag-deterministic-e63c6608


0it [00:00, ?it/s]

Experiment: <ExperimentResults roasted-wire-14>


## 5. Analyze results in a scorecard

In [6]:
rows = []
for item in experiment_results:
    question = item["example"].inputs["question"]
    answer_preview = (item["run"].outputs or {}).get("answer", "")[:120]
    for eval_result in item["evaluation_results"]["results"]:
        rows.append({
            "question": question[:60] + "...",
            "evaluator": eval_result.key,
            "score": eval_result.score,
            "comment": eval_result.comment,
            "answer_preview": answer_preview + "...",
        })

results_df = pd.DataFrame(rows)
display(results_df)

summary = results_df.groupby("evaluator")["score"].agg(["mean", "count"]).rename(columns={"mean": "avg_score"})
print("\n=== Score Summary ===")
display(summary)

overall_pass_rate = results_df["score"].mean()
print(f"\nOverall pass rate across all evaluators: {overall_pass_rate:.1%}")

,question,evaluator,score,comment,answer_preview
0,A customer was overcharged on their invoice. W...,topic_keyword,1.0,Topic found,"Based on the retrieved complaints, here is a c..."
1,A customer was overcharged on their invoice. W...,retrieval_relevance,1.0,"Checked aliases: ['refund', 'billing', 'overch...","Based on the retrieved complaints, here is a c..."
2,A customer was overcharged on their invoice. W...,answer_length,1.0,Word count: 94,"Based on the retrieved complaints, here is a c..."
3,My package arrived two weeks late. How was thi...,topic_keyword,1.0,Topic found,"Based on the retrieved complaints, here is a c..."
4,My package arrived two weeks late. How was thi...,retrieval_relevance,1.0,"Checked aliases: ['replacement', 'delivery', '...","Based on the retrieved complaints, here is a c..."
5,My package arrived two weeks late. How was thi...,answer_length,1.0,Word count: 94,"Based on the retrieved complaints, here is a c..."
6,The product I received was defective. What res...,topic_keyword,1.0,Topic found,"Based on the retrieved complaints, here is a c..."
7,The product I received was defective. What res...,retrieval_relevance,1.0,"Checked aliases: ['warranty', 'defective', 'pr...","Based on the retrieved complaints, here is a c..."
8,The product I received was defective. What res...,answer_length,1.0,Word count: 88,"Based on the retrieved complaints, here is a c..."
9,How do we handle subscription cancellation com...,topic_keyword,1.0,Topic found,"Based on the retrieved complaints, here is a c..."



=== Score Summary ===


,avg_score,count
evaluator,,
answer_length,1.0,5
retrieval_relevance,1.0,5
topic_keyword,1.0,5



Overall pass rate across all evaluators: 100.0%


## 6. LLM-as-judge with OpenEvals (optional)

When rule-based checks aren't enough, use an **LLM-as-judge**:
a separate LLM scores the answer on correctness, helpfulness, hallucination, etc.

OpenEvals provides pre-built prompts like `CORRECTNESS_PROMPT`.

> **Requires `OPENAI_API_KEY`** — skipped automatically if not set.

In [7]:
from openevals.llm import create_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT, CONCISENESS_PROMPT

if os.getenv("OPENAI_API_KEY"):
    correctness_judge = create_llm_as_judge(
        prompt=CORRECTNESS_PROMPT,
        model="openai:gpt-4o-mini",
        feedback_key="correctness",
    )

    def correctness_evaluator(run, example):
        return correctness_judge(
            inputs=example.inputs,
            outputs=run.outputs,
            reference_outputs={"answer": (example.outputs or {}).get("reference_answer", "")},
        )

    llm_examples = langsmith_examples[:2]  # limit cost for demo
    llm_experiment = evaluate(
        complaint_rag_target,
        data=llm_examples,
        evaluators=[correctness_evaluator],
        experiment_prefix="complaint-rag-llm-judge",
        upload_results=False,
    )

    llm_rows = []
    for item in llm_experiment:
        for eval_result in item["evaluation_results"]["results"]:
            llm_rows.append({
                "question": item["example"].inputs["question"][:50],
                "evaluator": eval_result.key,
                "score": eval_result.score,
                "comment": eval_result.comment,
            })
    display(pd.DataFrame(llm_rows))
else:
    print("Skipping LLM-as-judge — set OPENAI_API_KEY in .env to enable.")
    print("\nExample usage when key is available:")
    print("""
    correctness_judge = create_llm_as_judge(
        prompt=CORRECTNESS_PROMPT,
        model='openai:gpt-4o-mini',
        feedback_key='correctness',
    )
    result = correctness_judge(
        inputs={'question': '...'},
        outputs={'answer': '...'},
        reference_outputs={'answer': '...'},
    )
    """)

Skipping LLM-as-judge — set OPENAI_API_KEY in .env to enable.

Example usage when key is available:

    correctness_judge = create_llm_as_judge(
        prompt=CORRECTNESS_PROMPT,
        model='openai:gpt-4o-mini',
        feedback_key='correctness',
    )
    result = correctness_judge(
        inputs={'question': '...'},
        outputs={'answer': '...'},
        reference_outputs={'answer': '...'},
    )
    


## 7. Manual eval loop (no LangSmith dependency)

For quick iteration, you can run evals in a plain Python loop.
This is useful in CI or when you don't want any tracing overhead.

In [8]:
def run_manual_eval(examples, target_fn, evaluators):
    all_scores = {ev.__name__: [] for ev in evaluators}
    for ex in examples:
        outputs = target_fn(ex["inputs"])
        run = SimpleRun(outputs)
        example = SimpleExample(ex["outputs"])
        for ev in evaluators:
            result = ev(run, example)
            all_scores[ev.__name__].append(result["score"])
    return {name: sum(scores) / len(scores) for name, scores in all_scores.items()}

manual_summary = run_manual_eval(
    GOLDEN_EXAMPLES,
    complaint_rag_target,
    [topic_keyword_evaluator, retrieval_relevance_evaluator, answer_length_evaluator],
)

print("Manual eval pass rates:")
for name, rate in manual_summary.items():
    print(f"  {name}: {rate:.1%}")

Manual eval pass rates:
  topic_keyword_evaluator: 100.0%
  retrieval_relevance_evaluator: 100.0%
  answer_length_evaluator: 100.0%


## 8. Key takeaways

| Step | What you did |
|------|-------------|
| Dataset | Created golden examples with `inputs` + `reference_outputs` |
| Target | Wrapped `ComplaintRAG.answer()` as the function under test |
| Evaluators | Used OpenEvals + custom topic/retrieval/length checks |
| Orchestration | Ran `langsmith.evaluate()` end-to-end with `upload_results=False` |
| Analysis | Built a pandas scorecard with per-evaluator pass rates |
| LLM-as-judge | Optional OpenEvals `create_llm_as_judge` for semantic scoring |

### Next steps
- Add more edge-case examples (ambiguous queries, multi-topic complaints)
- Log experiments to [LangSmith](https://smith.langchain.com) with `LANGSMITH_API_KEY`
- Add RAG-specific evals: faithfulness, context precision (OpenEvals `HALLUCINATION_PROMPT`)
- Gate deployments on minimum pass rate (e.g. `topic_keyword >= 0.8`)